# Venturijeva cijev: predvidi → izračunaj → provjeri

Venturijeva cijev pretvara razliku tlakova u procjenu protoka. Ovdje tlak nije savršeno poznat broj, nego rezultat mjerenja. Cilj je odvojiti **modelsku pretpostavku** (stacionaran, nestlačiv tok; poznat koeficijent istjecanja) od **mjerne nesigurnosti**.

## Predvidi

Prije računa odgovori:

1. Ako se izmjerena razlika tlakova učetverostruči, koliko se promijeni protok?
2. Hoće li ista apsolutna pogreška promjera više utjecati na ulazni promjer ili na grlo?
3. Može li diferencijalni senzor sam dokazati da nigdje nema kavitacije?

Posljednje pitanje je važno: ovaj model koristi samo razliku tlakova. Za kavitaciju treba zasebno poznavati **apsolutni** lokalni tlak i tlak para.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def venturi_flow(D1, D2, delta_p, rho, Cd):
    # Volumni protok iz diferencijalnog tlaka; sve veličine su u SI.
    D1, D2, delta_p, rho, Cd = np.broadcast_arrays(D1, D2, delta_p, rho, Cd)
    if np.any(D2 <= 0) or np.any(D1 <= D2) or np.any(delta_p < 0):
        raise ValueError("Za ovaj model treba vrijediti D1 > D2 > 0 i Δp ≥ 0.")
    A2 = np.pi * D2**2 / 4
    beta = D2 / D1
    return Cd * A2 * np.sqrt(2 * delta_p / (rho * (1 - beta**4)))

base = dict(D1=0.080, D2=0.040, delta_p=18_000.0, rho=998.0, Cd=0.985)
sigma = dict(D1=0.00015, D2=0.00010, delta_p=80.0, rho=1.0, Cd=0.003)

Q = float(venturi_flow(**base))
A1, A2 = np.pi*base["D1"]**2/4, np.pi*base["D2"]**2/4
v1, v2 = Q/A1, Q/A2
print(f"Q = {1e3*Q:.3f} L/s; v1 = {v1:.3f} m/s; v2 = {v2:.3f} m/s")


## Izračunaj: propagacija senzorske nesigurnosti

Za međusobno neovisne male nesigurnosti prva procjena je

\[
u_Q^2 \approx \sum_i\left(\frac{\partial Q}{\partial x_i}u_{x_i}\right)^2.
\]

Derivacije računamo centriranom razlikom, a rezultat neovisno provjeravamo determinističkim Monte Carlo uzorkovanjem. Slučajni generator ima fiksno sjeme pa je rezultat ponovljiv i u pregledniku.


In [ ]:
def centered_derivative(key, values, scales):
    step = max(scales[key] * 1e-3, abs(values[key]) * 1e-8)
    plus, minus = values.copy(), values.copy()
    plus[key] += step
    minus[key] -= step
    return (float(venturi_flow(**plus)) - float(venturi_flow(**minus))) / (2*step)

gradient = {key: centered_derivative(key, base, sigma) for key in base}
variance_terms = {key: (gradient[key]*sigma[key])**2 for key in base}
u_linear = np.sqrt(sum(variance_terms.values()))

rng = np.random.default_rng(20260801)
n_samples = 50_000
samples = {
    key: rng.normal(base[key], sigma[key], n_samples)
    for key in base
}
Q_mc = venturi_flow(**samples)
u_mc = np.std(Q_mc, ddof=1)
q025, q975 = np.quantile(Q_mc, [0.025, 0.975])

print(f"Linearna standardna nesigurnost: {1e3*u_linear:.4f} L/s")
print(f"Monte Carlo standardna nesigurnost: {1e3*u_mc:.4f} L/s")
print(f"Monte Carlo 95 %-tni interval: [{1e3*q025:.3f}, {1e3*q975:.3f}] L/s")
for key, term in sorted(variance_terms.items(), key=lambda item: -item[1]):
    print(f"  {key:7s}: {100*term/u_linear**2:5.1f} % varijance")


## Provjeri

Tri provjere koriste informacije koje nisu ugrađene u isti numerički korak:

- povratnim modelom izračunavamo tlak iz dobivenog protoka;
- provjeravamo poznati zakon skaliranja \(Q\propto\sqrt{\Delta p}\);
- uspoređujemo linearnu propagaciju s Monte Carlo uzorkovanjem.


In [ ]:
beta = base["D2"]/base["D1"]
delta_p_back = 0.5*base["rho"]*(1-beta**4)*(Q/(base["Cd"]*A2))**2
Q_four_dp = float(venturi_flow(**{**base, "delta_p": 4*base["delta_p"]}))

assert np.isclose(delta_p_back, base["delta_p"], rtol=1e-12)
assert np.isclose(Q_four_dp/Q, 2.0, rtol=1e-12)
assert abs(u_mc/u_linear - 1) < 0.05

labels = list(variance_terms)
shares = np.array([variance_terms[k] for k in labels])/u_linear**2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(1e3*Q_mc, bins=55, color="#7cb5d6", edgecolor="white")
axes[0].axvline(1e3*Q, color="#b43c35", lw=2, label="nominalno")
axes[0].set(xlabel="Q (L/s)", ylabel="broj uzoraka", title="Propagirana nesigurnost")
axes[0].legend()
axes[1].bar(labels, 100*shares, color="#256d85")
axes[1].set(ylabel="udio u varijanci Q (%)", title="Osjetljivost na ulaze")
axes[1].tick_params(axis="x", rotation=30)
for ax in axes: ax.grid(True, axis="y", ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Ponovi račun s dvostruko većom nesigurnošću \(D_2\). Je li racionalnije poboljšati senzor tlaka ili mjerenje promjera? Zaključak vrijedi samo unutar navedenog modela i raspona; nesigurnost koeficijenta \(C_d\) predstavlja dio kalibracije koji idealna Bernoullijeva jednadžba sama ne može odrediti.
